# Typing module: Annotated, operator

`Annotated` attaches metadata to a type without changing the underlying type used by static type checkers. Frameworks or other code can inspect that metadata.

`TypedDict` describes the expected keys and value types of a dictionary for static type checking. It does not validate dictionary contents at runtime.

`operator` is a standard-library module with function forms of Python operators. For example, `operator.add(a, b)` does the same operation as `a + b`.

```python
from typing import Annotated, TypedDict
import operator

class Person(TypedDict):
    name: Annotated[str, "display name"]
    age: int

sum_of_two_numbers = operator.add(2, 3)  # 5
```

In [36]:
# Typing sample 02
from typing import Union, Optional, Callable, Annotated
import operator
from pandas import DataFrame
from pydantic import BaseModel, Field, ValidationError, field_validator


In [2]:
def get_user_details(user_id: Union[int, str], transform: Optional[bool] = None) -> dict:
    print("Type of user_id:", type(user_id))
    if transform:
        return {"user_id": str(user_id), "name": "John Doe", "age": 30}
    return {"user_id": user_id, "name": "John Doe", "age": 30}

print(get_user_details(123))
print(get_user_details("abc"))

Type of user_id: <class 'int'>
{'user_id': 123, 'name': 'John Doe', 'age': 30}
Type of user_id: <class 'str'>
{'user_id': 'abc', 'name': 'John Doe', 'age': 30}


In [3]:
def exec_operation(operation: Callable[[int, int], int], a: int, b: int) -> int:
    return operation(a, b)

def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b    

print("Executing operations:")
print("- add: ")
print(exec_operation(add, 3, 5))
print("- multiply: ")
print(exec_operation(multiply, 3, 5))

print("- operator.add: ")
print(exec_operation(operator.add, 3, 5))
print("- operator.mul: ")
print(exec_operation(operator.mul, 3, 5))


Executing operations:
- add: 
8
- multiply: 
15
- operator.add: 
8
- operator.mul: 
15


In [12]:
students = [("John Doe", 20, "Computer Science"),
            ("Jane Smith", 22, "Mathematics"),
            ("Alice Johnson", 19, "Physics")]

print("First student's name:")
print(operator.itemgetter(0)(students[0]))

print("First student's age:")
print(operator.itemgetter(1)(students[0]))

print("First student's major:")
print(operator.itemgetter(2)(students[0]))

df_students = DataFrame(students)
print(df_students)

print("Avg. degrees: ")
print(sum(operator.itemgetter(1)(student) for student in students) / len(students))

print("Avg. degrees (from DataFrame): ")
print(df_students[1].mean())


First student's name:
John Doe
First student's age:
20
First student's major:
Computer Science
               0   1                 2
0       John Doe  20  Computer Science
1     Jane Smith  22       Mathematics
2  Alice Johnson  19           Physics
Avg. degrees: 
20.333333333333332
Avg. degrees (from DataFrame): 
20.333333333333332


In [15]:
class Person:
    def __init__(self, name: str, age: int, major: str):
        self.name = name
        self.age = age
        self.major = major
 
persons = [
    Person("John Doe", 20, "Computer Science"),
    Person("Jane Smith", 22, "Mathematics"),
    Person("Alice Johnson", 19, "Physics")
]

print("First person's name:")
print(operator.attrgetter("name")(persons[0]))

print("Avg. age:")
print(sum(operator.attrgetter("age")(person) for person in persons) / len(persons))



First person's name:
John Doe
Avg. age:
20.333333333333332


In [ ]:
# Annotated

UserId = Annotated[int | str, "User ID", {"required": True}]

def get_user_id(user_id: UserId) -> UserId:
    if not isinstance(user_id, (int, str)):
        raise ValueError("Invalid user ID")
    print(f"User ID: {user_id}")
    return user_id

print(get_user_id(123))
print(get_user_id("abc"))
# print(get_user_id(12.34))  # This should raise a ValueError
# print(get_user_id(None))  # This should raise a ValueError

class User:
    user_id: UserId
    username: Annotated[str, "Username", {"required": True, "min_length": 5, "max_length": 10}]
    
    def __init__(self, user_id: UserId, username: str):
        self.user_id = user_id
        self.username = username
    
    def __repr__(self):
        return f"User(user_id={self.user_id}, username={self.username})"

print(User(user_id=123, username="username"))
print(User(user_id=123, username="user"))  # This should raise an error due to min_length constraint
print(User(user_id=12.3, username="username12345"))  # This should raise an error due to max_length constraint


Attributes = Annotated[list[str], operator.add, {"max_length": 4}]

attrs: Attributes = ["attribute1", "attribute2"]
attrs.append("attr_new")
print(attrs)

attrs.append("attr_new1")
attrs.append("attr_new2")
print(attrs)



User ID: 123
123
User ID: abc
abc
User(user_id=123, username=username)
User(user_id=123, username=user)
User(user_id=12.3, username=username12345)
['attribute1', 'attribute2', 'attr_new']
['attribute1', 'attribute2', 'attr_new', 'attr_new1', 'attr_new2']


In [44]:
# pydantic usage example

class UserRegistrationForm(BaseModel):
    user_id: UserId
    username: str = Field(..., min_length=5, max_length=20)
    email: Annotated[str, "Email", Field(title="E-Mail validator", pattern=r"[^@]+@[^@]+\.[^@]+")]

    @field_validator("username")
    @classmethod
    def validate_username(cls, value):
        if " " in value:
            raise ValueError("Username must not contain spaces")
        return value

    @field_validator("user_id")
    @classmethod
    def validate_user_id(cls, value):
        if value <= 0:
            raise ValueError("User ID must be greater than 0")
        return value

# Example usage
try:
    form = UserRegistrationForm(user_id=1, username="valid_user", email="valid@example.com")
    print(form)
except ValidationError as e:
    print(e)
    
# Invalid usage example
try:
    form = UserRegistrationForm(user_id=-1, username="invalid user", email="#_invalid_example.com")
    print(form)
except ValidationError as e:
    print(e)



user_id=1 username='valid_user' email='valid@example.com'
3 validation errors for UserRegistrationForm
user_id
  Value error, User ID must be greater than 0 [type=value_error, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
username
  Value error, Username must not contain spaces [type=value_error, input_value='invalid user', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error
email
  String should match pattern '[^@]+@[^@]+\.[^@]+' [type=string_pattern_mismatch, input_value='#_invalid_example.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_pattern_mismatch
